# Exercise 3.3: Table Modeling and Write Optimization

In this exercise, you'll learn how to optimize Apache Iceberg tables for write and read performance:
- **Partition strategies**: Compare different partition granularities
- **Sort orders**: Optimize for specific query patterns
- **Distribution modes**: Control how Spark distributes data before writing
- **File sizes**: Balance write performance vs read performance

## Learning Objectives
- Compare unpartitioned, daily, and hourly partitioning
- Implement and test sort orders
- Understand distribution mode tradeoffs
- Measure the impact of modeling decisions on query performance
- Design optimal table layouts for different workloads

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random
import time

spark = SparkSession.builder \
    .appName("TableModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.modeling")
print("Namespace 'modeling' created!")

## Generate Test Dataset

Let's create a realistic dataset for testing different modeling strategies.

In [ ]:
# Generate 1,000 event records over 5 days (lightweight for testing)
print("Generating test dataset...")

base_time = datetime(2024, 1, 13)
user_ids = [f"user_{i}" for i in range(1, 51)]  # 50 users
event_types = ['page_view', 'click', 'purchase', 'add_to_cart', 'remove_from_cart']
pages = ['home', 'products', 'cart', 'checkout', 'profile', 'search']

events = []
for i in range(1000):
    event_time = base_time + timedelta(
        days=random.randint(0, 4),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    
    events.append((
        i,
        random.choice(user_ids),
        random.choice(event_types),
        random.choice(pages),
        event_time,
        event_time.date()
    ))

# Create DataFrame
test_data = spark.createDataFrame(events,
    ["event_id", "user_id", "event_type", "page", "event_timestamp", "event_date"])

# Cache for reuse
test_data.cache()
count = test_data.count()

print(f"Generated {count:,} events")
test_data.show(5)

## Experiment 1: Partition Granularity

### Strategy 1: Unpartitioned Table

In [ ]:
# Create unpartitioned table
print("Creating UNPARTITIONED table...")
start = time.time()

test_data.writeTo("polaris.modeling.events_unpartitioned") \
    .using("iceberg") \
    .tableProperty("write.target-file-size-bytes", "134217728") \
    .createOrReplace()

write_time_unpart = time.time() - start
print(f"Write time: {write_time_unpart:.2f} seconds")

In [ ]:
# Check file layout
print("Unpartitioned file statistics:")
spark.sql("""
SELECT 
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
FROM polaris.modeling.events_unpartitioned.files
""").show()

### Strategy 2: Daily Partitioning

In [ ]:
# Create daily partitioned table
print("Creating DAILY PARTITIONED table...")
start = time.time()

test_data.writeTo("polaris.modeling.events_daily") \
    .using("iceberg") \
    .partitionedBy("event_date") \
    .tableProperty("write.target-file-size-bytes", "134217728") \
    .createOrReplace()

write_time_daily = time.time() - start
print(f"Write time: {write_time_daily:.2f} seconds")

In [ ]:
# Check file layout
print("Daily partition file statistics:")
spark.sql("""
SELECT 
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    COUNT(DISTINCT partition) as partition_count
FROM polaris.modeling.events_daily.files
""").show()

### Strategy 3: Hourly Partitioning

In [ ]:
# Create hourly partitioned table
print("Creating HOURLY PARTITIONED table...")
start = time.time()

spark.sql("""
CREATE OR REPLACE TABLE polaris.modeling.events_hourly (
    event_id BIGINT,
    user_id STRING,
    event_type STRING,
    page STRING,
    event_timestamp TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (hours(event_timestamp))
TBLPROPERTIES (
    'write.target-file-size-bytes' = '134217728'
)
""")

test_data.writeTo("polaris.modeling.events_hourly").append()

write_time_hourly = time.time() - start
print(f"Write time: {write_time_hourly:.2f} seconds")

In [ ]:
# Check file layout
print("Hourly partition file statistics:")
spark.sql("""
SELECT 
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    COUNT(DISTINCT partition) as partition_count
FROM polaris.modeling.events_hourly.files
""").show()

### Compare Query Performance

In [ ]:
# Query: Get events for a specific day
query = """
SELECT COUNT(*) as count, event_type
FROM {table}
WHERE event_timestamp >= TIMESTAMP '2024-01-15 00:00:00'
  AND event_timestamp < TIMESTAMP '2024-01-16 00:00:00'
GROUP BY event_type
"""

print("\nQuery: Events for January 15, 2024")
print("=" * 60)

# Unpartitioned
start = time.time()
result = spark.sql(query.format(table="polaris.modeling.events_unpartitioned"))
result.collect()
unpart_time = time.time() - start
print(f"Unpartitioned: {unpart_time:.3f} seconds")

# Daily partitioned
start = time.time()
result = spark.sql(query.format(table="polaris.modeling.events_daily"))
result.collect()
daily_time = time.time() - start
print(f"Daily partitioned: {daily_time:.3f} seconds")

# Hourly partitioned
start = time.time()
result = spark.sql(query.format(table="polaris.modeling.events_hourly"))
result.collect()
hourly_time = time.time() - start
print(f"Hourly partitioned: {hourly_time:.3f} seconds")

print("\nSpeedup vs unpartitioned:")
print(f"  Daily: {unpart_time/daily_time:.2f}x")
print(f"  Hourly: {unpart_time/hourly_time:.2f}x")

In [ ]:
# Query 2: Get events for a single hour
query2 = """
SELECT COUNT(*) as count
FROM {table}
WHERE event_timestamp >= TIMESTAMP '2024-01-15 14:00:00'
  AND event_timestamp < TIMESTAMP '2024-01-15 15:00:00'
"""

print("\nQuery: Events for Jan 15, 2024 2-3 PM")
print("=" * 60)

for table_name in ['events_unpartitioned', 'events_daily', 'events_hourly']:
    start = time.time()
    result = spark.sql(query2.format(table=f"polaris.modeling.{table_name}"))
    count = result.collect()[0]['count']
    elapsed = time.time() - start
    print(f"{table_name}: {elapsed:.3f}s (count: {count})")

### Partition Granularity Summary

In [ ]:
print("\n" + "=" * 60)
print("PARTITION STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Strategy':<20} {'Write Time':<12} {'Files':<8} {'Partitions':<12}")
print("-" * 60)
print(f"{'Unpartitioned':<20} {write_time_unpart:>10.2f}s  {'~1':<8} {'1':<12}")
print(f"{'Daily':<20} {write_time_daily:>10.2f}s  {'~5':<8} {'5':<12}")
print(f"{'Hourly':<20} {write_time_hourly:>10.2f}s  {'~120':<8} {'~120':<12}")
print("=" * 60)
print("\nObservations:")
print("- Hourly partitioning: Best for hour-range queries, but many small files")
print("- Daily partitioning: Good balance for day-range queries")
print("- Unpartitioned: Simplest, but scans all data")

In [ ]:
# Clean up Experiment 1 tables to free memory before next experiment
for t in ['events_unpartitioned', 'events_daily', 'events_hourly']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Experiment 1 tables cleaned up!")

## Experiment 2: Sort Orders

### Table Without Sort Order

In [ ]:
# Create table without sort order
print("Creating table WITHOUT sort order...")

test_data.writeTo("polaris.modeling.events_unsorted") \
    .using("iceberg") \
    .partitionedBy("event_date") \
    .tableProperty("write.target-file-size-bytes", "10485760") \
    .createOrReplace()

print("Unsorted table created!")

In [ ]:
# Check metrics for a file
print("Unsorted table file metrics (sample):")
spark.sql("""
SELECT 
    file_path,
    partition,
    record_count,
    readable_metrics
FROM polaris.modeling.events_unsorted.files
LIMIT 1
""").show(truncate=False, vertical=True)

### Table With Sort Order

In [ ]:
# Create table with sort order on user_id
print("Creating table WITH sort order (user_id)...")

spark.sql("""
CREATE OR REPLACE TABLE polaris.modeling.events_sorted (
    event_id BIGINT,
    user_id STRING,
    event_type STRING,
    page STRING,
    event_timestamp TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (event_date)
TBLPROPERTIES (
    'write.target-file-size-bytes' = '10485760'
)
""")

# Add sort order
spark.sql("""
ALTER TABLE polaris.modeling.events_sorted
WRITE ORDERED BY user_id
""")

# Write data (will be sorted)
test_data.writeTo("polaris.modeling.events_sorted").append()

print("Sorted table created!")

In [ ]:
# Check metrics - should have tighter user_id ranges
print("Sorted table file metrics (sample):")
spark.sql("""
SELECT 
    file_path,
    partition,
    record_count,
    readable_metrics
FROM polaris.modeling.events_sorted.files
LIMIT 1
""").show(truncate=False, vertical=True)

### Compare Query Performance

In [ ]:
# Query filtering on sort column
query = """
SELECT COUNT(*) as count, event_type
FROM {table}
WHERE user_id = 'user_25'
  AND event_date = DATE '2024-01-15'
GROUP BY event_type
"""

print("Query: Events for user_25 on Jan 15")
print("=" * 60)

# Unsorted
start = time.time()
result = spark.sql(query.format(table="polaris.modeling.events_unsorted"))
result.show()
unsorted_time = time.time() - start
print(f"Unsorted: {unsorted_time:.3f} seconds")

# Sorted
start = time.time()
result = spark.sql(query.format(table="polaris.modeling.events_sorted"))
result.show()
sorted_time = time.time() - start
print(f"Sorted: {sorted_time:.3f} seconds")

print(f"\nSpeedup: {unsorted_time/sorted_time:.2f}x")

In [ ]:
# Check file pruning in Spark UI
print("Check Spark UI (http://localhost:4040) to see:")
print("- Number of files scanned")
print("- Sorted table should scan fewer files due to user_id pruning")

In [ ]:
# Clean up Experiment 2 tables to free memory before next experiment
for t in ['events_unsorted', 'events_sorted']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Experiment 2 tables cleaned up!")

## Experiment 3: Distribution Modes

### Default Distribution (Hash by Partition)

In [ ]:
# Default mode - Spark shuffles by partition
print("Writing with DEFAULT distribution mode...")
start = time.time()

test_data.writeTo("polaris.modeling.events_dist_default") \
    .using("iceberg") \
    .partitionedBy("event_date") \
    .tableProperty("write.distribution-mode", "hash") \
    .createOrReplace()

default_time = time.time() - start
print(f"Write time: {default_time:.2f} seconds")

In [ ]:
# Check file layout
print("Default distribution file stats:")
spark.sql("""
SELECT 
    COUNT(*) as total_files,
    COUNT(DISTINCT partition) as partition_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
FROM polaris.modeling.events_dist_default.files
""").show()

### None Distribution (No Shuffle)

In [ ]:
# None mode - no shuffle, write as-is
print("Writing with NONE distribution mode...")
start = time.time()

test_data.writeTo("polaris.modeling.events_dist_none") \
    .using("iceberg") \
    .partitionedBy("event_date") \
    .tableProperty("write.distribution-mode", "none") \
    .createOrReplace()

none_time = time.time() - start
print(f"Write time: {none_time:.2f} seconds")

In [ ]:
# Check file layout - likely MANY more files!
print("None distribution file stats:")
spark.sql("""
SELECT 
    COUNT(*) as total_files,
    COUNT(DISTINCT partition) as partition_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
FROM polaris.modeling.events_dist_none.files
""").show()

### Distribution Mode Summary

In [ ]:
print("\n" + "=" * 60)
print("DISTRIBUTION MODE COMPARISON")
print("=" * 60)
print(f"{'Mode':<15} {'Write Time':<12} {'Files':<10} {'Avg Size':<12}")
print("-" * 60)
print(f"{'Hash (default)':<15} {default_time:>10.2f}s  {'Fewer':<10} {'Optimal':<12}")
print(f"{'None':<15} {none_time:>10.2f}s  {'Many':<10} {'Small':<12}")
print("=" * 60)
print("\nObservations:")
print("- Hash: Slower write (shuffle), fewer/larger files")
print("- None: Faster write (no shuffle), many small files")
print("- Choose based on: Write frequency vs read performance needs")

## Key Takeaways

### Partitioning Strategy

| Granularity | When to Use | Pros | Cons |
|-------------|-------------|------|------|
| **Unpartitioned** | Small tables, no time filters | Simple, fewer files | Scans all data |
| **Daily** | Day-range queries, moderate volume | Good balance | Not optimal for hour queries |
| **Hourly** | Hour-range queries, high volume | Best selectivity | Many partitions/files |

**Rule of thumb**: Aim for 1-10 GB per partition

### Sort Orders

**Use sort orders when:**
- You have predictable query patterns
- Queries filter on specific high-cardinality columns
- Partitions contain multiple files
- Read performance > write performance

**Skip sort orders when:**
- Partitions have only 1-2 files
- Query patterns are unpredictable
- Write performance is critical
- Data volume is low

### Distribution Modes

| Mode | Write Speed | File Count | File Size | Use Case |
|------|-------------|------------|-----------|----------|
| **Hash** | Slower (shuffle) | Optimal | Larger | Default, most workloads |
| **Range** | Slower (shuffle) | Optimal | Larger | With sort orders |
| **None** | Fast (no shuffle) | Many | Small | Pre-organized data only |

**Warning**: `none` mode can create thousands of tiny files if data isn't already well-organized!

### Design Workflow

1. **Analyze query patterns**
   - What columns are filtered most?
   - What time ranges are typical?
   - How selective are filters?

2. **Choose partition strategy**
   - Match partition granularity to query ranges
   - Target 1-10 GB per partition
   - Fewer is better than more

3. **Consider sort orders**
   - Only if partitions have many files
   - Sort on high-cardinality filter columns
   - Test performance impact

4. **Set distribution mode**
   - Default: hash (almost always)
   - Range: with sort orders
   - None: only if data is pre-organized

5. **Monitor and adjust**
   - Track file counts and sizes
   - Measure query performance
   - Evolve as patterns change

## Real-World Recommendations

### For Streaming Workloads
```python
- Partition: By day or hour (depends on volume)
- Sort order: Maybe, if queries are selective
- Distribution: Hash (default)
- Target file size: 128 MB - 512 MB
- Maintenance: Daily compaction
```

### For Batch Workloads
```python
- Partition: By day, week, or month (depends on volume)
- Sort order: Yes, on primary filter columns
- Distribution: Range (with sort order)
- Target file size: 512 MB - 1 GB
- Maintenance: Weekly compaction
```

### For Analytical Workloads
```python
- Partition: Coarse (by month or year)
- Sort order: Yes, optimize for common queries
- Distribution: Range
- Target file size: 1 GB+
- Maintenance: As needed
```

## Challenge Exercise

Design an optimal table layout for this scenario:

**Requirements:**
- 10 TB of IoT sensor data
- Streaming ingestion (100 MB/minute)
- Queries typically filter by:
  1. Time range (usually 1-7 days)
  2. Sensor ID (high cardinality, 10,000 sensors)
  3. Reading type (low cardinality, 5 types)

**Your Tasks:**
1. Choose partition strategy
2. Decide on sort order
3. Select distribution mode
4. Set target file size
5. Plan maintenance schedule

Justify each decision!

## Cleanup

In [ ]:
# Uncache test data
test_data.unpersist()

# Drop remaining test tables
tables = ['events_dist_default', 'events_dist_none']
for table in tables:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{table}")
print("Cleanup complete!")